# Can we predict `passed_quiz`?
Follows `checklist.md`.

**Setup decisions**
- There is no row-level timestamp, so there is no "later rows" test set. `student_key` repeats across rows (~7 sessions each), so the test set holds **whole students** (checklist 1) and CV uses whole-student folds (2).
- Features: `enrol_year`, `school_code` (categorical), `prior_score` (placement test, taken before the sessions), `session_minutes`, `hints_used`. I assume the prediction is made at the end of the session, just before the closing quiz, so in-session columns are known. If a prediction is needed before the session starts, drop those two. `student_key` is an identifier: used for grouping only, never as a feature.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
df = pd.read_csv('data.csv')
print(df.shape, df.passed_quiz.mean())
print('share of students whose prior_score varies across rows:', (df.groupby('student_key').prior_score.nunique() > 1).mean())

## 1. Hold out the test set (whole students); not used until step 6

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr, te = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.passed_quiz.mean())
FEATS = ['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']
Xtr, ytr, gtr = train[FEATS], train.passed_quiz, train.student_key

## 4. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence.
2. `logreg_numeric`: logistic regression on the numeric features (scaled).
3. `logreg_school`: same plus one-hot `school_code`.
4. `hgb_d{2,3}_lr{0.05,0.1}`: gradient boosting, four hyperparameter settings, `school_code` as categorical.

Primary metric: ROC AUC. Decision threshold for accuracy/F1 is chosen on CV out-of-fold predictions by maximising F1. Preprocessing lives inside pipelines, so it is refit in every fold (3).

In [ ]:
num = ['enrol_year', 'prior_score', 'session_minutes', 'hints_used']
def lr(cat):
    ct = ColumnTransformer([('n', StandardScaler(), num)] + ([('c', OneHotEncoder(handle_unknown='ignore'), ['school_code'])] if cat else []))
    return Pipeline([('p', ct), ('m', LogisticRegression(max_iter=1000))])
def hgb(d, l):
    return Pipeline([('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=l, max_iter=200,
        categorical_features=[FEATS.index('school_code')], random_state=0))])
cands = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))]),
         'logreg_numeric': lr(False), 'logreg_school': lr(True)}
for d in (2, 3):
    for l in (0.05, 0.1):
        cands[f'hgb_d{d}_lr{l}'] = hgb(d, l)
folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
oof, rows = {}, []
for k, m in cands.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m.fit(Xtr.iloc[a], ytr.iloc[a]); p[b] = m.predict_proba(Xtr.iloc[b])[:, 1]
        aucs.append(roc_auc_score(ytr.iloc[b], p[b]))
    oof[k] = p; rows.append((k, np.mean(aucs), np.std(aucs)))
cvres = pd.DataFrame(rows, columns=['model', 'cv_auc_mean', 'cv_auc_sd']).sort_values('cv_auc_mean', ascending=False)
cvres

## 5. Choose model and threshold on CV only

In [ ]:
best = cvres.iloc[0].model
ths = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(ytr, oof[best] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr, 2))
print('CV accuracy at threshold:', accuracy_score(ytr, oof[best] >= thr), '| CV F1:', max(f1s))
print('CV accuracy of always predicting 0:', 1 - ytr.mean())

**Choice (fixed before the test set is scored):** the model and threshold printed above. Nothing is changed after seeing the test score.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = cands[best].fit(Xtr, ytr)
pt = final.predict_proba(test[FEATS])[:, 1]; yt = test.passed_quiz
auc = roc_auc_score(yt, pt)
rng = np.random.default_rng(0)
idx = test.groupby('student_key').indices; ids = list(idx)
bs = []
for _ in range(1000):
    ix = np.concatenate([idx[s] for s in rng.choice(ids, len(ids))])
    bs.append(roc_auc_score(yt.iloc[ix], pt[ix]))
print(f'TEST AUC {auc:.3f}  95% student-bootstrap CI [{np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f}]')
print('TEST accuracy', accuracy_score(yt, pt >= thr), '| F1', f1_score(yt, pt >= thr), '| Brier', brier_score_loss(yt, pt))
print('accuracy of always predicting 0:', 1 - yt.mean())

## Conclusion
The headline is the test AUC above with its student-level bootstrap interval. An AUC near 0.5 would mean these features carry little signal.